# 6. Two-Optimizer Adversarial cVAE + MMD Batch Correction**Ключевые отличия от notebook 5 (single-optimizer GRL):**- ✅ Два отдельных optimizer (encoder+decoder vs discriminator)- ✅ 3 шага discriminator на 1 шаг encoder (стандартный GAN-подход)- ✅ Усиленный discriminator (256→128, spectral normalization, LeakyReLU)- ✅ MMD loss как дополнительный/альтернативный regularizer- ✅ `loss_type` ∈ {'adversarial', 'mmd', 'both'}- ✅ Полные scib метрики: kBET, iLISI, ASW_batch, cLISI, Sil_bio, NMI, ARI- ✅ Comparison table: Raw vs KL-cVAE vs Adv2-cVAE (adversarial / mmd / both)

In [ ]:
# Cell 1: Setup & Installimport subprocess, sys, os# In Colab: clone/pull repoif "google.colab" in sys.modules:    repo = "/content/batchcor-rna-embeds"    if not os.path.isdir(repo):        subprocess.run(["git", "clone", "-b", "feat/scvi-batch-correction",                        "https://github.com/onion-42/batchcor-rna-embeds.git", repo], check=True)    else:        subprocess.run(["git", "-C", repo, "pull", "origin", "feat/scvi-batch-correction"], check=True)    os.chdir(repo)    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)    subprocess.run(["uv", "pip", "install", "--system", "-q", "-e", "."], check=True)    subprocess.run(["uv", "pip", "install", "--system", "-q",                     "scib-metrics", "leidenalg", "igraph"], check=True)    from google.colab import drive    drive.mount("/content/drive", force_remount=False)

In [ ]:
# Cell 2: Importsimport warnings; warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport anndata as adimport scanpy as scfrom pathlib import Pathfrom loguru import loggerfrom sklearn.metrics import silhouette_scorefrom batchcor_rna_emb.logging_config import set_loggerfrom batchcor_rna_emb.data_io import load_cohortfrom batchcor_rna_emb.config import (    BATCH_COL, DIAGNOSIS_COL,    COMPASS_PT_EMBEDDING_KEY,    DATA_INTERIM_PT_DIR,)from batchcor_rna_emb.batch_correction.scvi_corrector import (    CVAEAdv2Config, CVAEAdv2Corrector,    CVAEConfig, CVAECorrector,)from batchcor_rna_emb.metrics.batch_metrics import (    compute_kbet, compute_ilisi, compute_asw_batch, compute_graph_connectivity,)from batchcor_rna_emb.metrics.bio_metrics import (    compute_clisi, compute_silhouette_bio, compute_nmi, compute_ari,    run_leiden_clustering,)from batchcor_rna_emb.metrics.aggregation import geometric_mean, build_comparison_tableset_logger("INFO")logger.info("Imports OK")

In [ ]:
# Cell 3: Paths & ConfigEMB_KEY = COMPASS_PT_EMBEDDING_KEY  # 4224-dimADV2_KEY = "FM_COMPASS_PT_emb_Adv2_corrected"# Data paths — adjust for Colab Driveif "google.colab" in __import__("sys").modules:    DRIVE_DATA = Path("/content/drive/MyDrive/batchcor-rna-embeds/data")    INTERIM_PT = DRIVE_DATA / "interim_PT"else:    INTERIM_PT = DATA_INTERIM_PT_DIR# Cohort sourcesCOHORTS = {    "PUB_KIRC_ICI_combined": INTERIM_PT / "PUB_KIRC_ICI_combined.adata.zarr",    "NSCLC_Tissue_ICI_Pred": INTERIM_PT / "NSCLC_Tissue_ICI_Pred.adata.zarr",    "PUB_BLCA_Mariathasan_EGAS00001002556": INTERIM_PT / "PUB_BLCA_Mariathasan_EGAS00001002556.adata.zarr",    "PUB_ccRCC_Immotion150_and_151_ICI": INTERIM_PT / "PUB_ccRCC_Immotion150_and_151_ICI.adata.zarr",    "PUB_ccRCC_Immotion150_and_151_TKI": INTERIM_PT / "PUB_ccRCC_Immotion150_and_151_TKI.adata.zarr",}# Only correct multi-batch cohortsMULTI_BATCH = ["PUB_KIRC_ICI_combined", "NSCLC_Tissue_ICI_Pred"]# Three loss variants to compareADV2_CONFIGS = {    "Adv2_adversarial": CVAEAdv2Config(loss_type="adversarial", lambda_adv=1.0, lambda_mmd=0.0),    "Adv2_mmd":         CVAEAdv2Config(loss_type="mmd",         lambda_adv=0.0, lambda_mmd=1.0),    "Adv2_both":        CVAEAdv2Config(loss_type="both",        lambda_adv=1.0, lambda_mmd=0.5),}# KL-cVAE baseline for comparisonKL_CONFIG = CVAEConfig(latent_dim=128, normalize=True, n_epochs=150)logger.info("Config ready. Multi-batch cohorts: {}", MULTI_BATCH)

In [ ]:
# Cell 4: Load all cohortsall_adata = {}for name, path in COHORTS.items():    all_adata[name] = load_cohort(path)    logger.info("  {} -> {} samples, {} batches, {} diagnoses",                name, all_adata[name].n_obs,                all_adata[name].obs[BATCH_COL].nunique(),                all_adata[name].obs[DIAGNOSIS_COL].nunique())

In [ ]:
# Cell 5: Helper functionsdef run_correction(adata, emb_key, batch_col, corrector):    """Fit corrector on train data, transform all."""    X = np.asarray(adata.obsm[emb_key]).astype(np.float32)    batches = adata.obs[batch_col].astype(str).values    corrected = corrector.fit_transform(X, batches)    return corrected, correctordef compute_all_metrics(adata, emb_key, batch_col, bio_col):    """Compute full scib metrics suite. Returns dict."""    n_batch = adata.obs[batch_col].nunique()    n_bio = adata.obs[bio_col].nunique()    m = {}    # Batch metrics (require >=2 batches)    if n_batch >= 2:        try: m["kBET"] = compute_kbet(adata, emb_key, batch_col)        except Exception as e: logger.warning("kBET failed: {}", e); m["kBET"] = np.nan        try: m["iLISI"] = compute_ilisi(adata, emb_key, batch_col)        except Exception as e: logger.warning("iLISI failed: {}", e); m["iLISI"] = np.nan        try: m["ASW_batch"] = compute_asw_batch(adata, emb_key, batch_col)        except Exception as e: logger.warning("ASW_batch failed: {}", e); m["ASW_batch"] = np.nan        try: m["graph_conn"] = compute_graph_connectivity(adata, batch_col, emb_key=emb_key)        except Exception as e: logger.warning("graph_conn failed: {}", e); m["graph_conn"] = np.nan    else:        m["kBET"] = m["iLISI"] = m["ASW_batch"] = m["graph_conn"] = np.nan    # Bio metrics (require >=2 labels)    if n_bio >= 2:        try: m["cLISI"] = compute_clisi(adata, emb_key, bio_col)        except Exception as e: logger.warning("cLISI failed: {}", e); m["cLISI"] = np.nan        try: m["Sil_bio"] = compute_silhouette_bio(adata, emb_key, bio_col)        except Exception as e: logger.warning("Sil_bio failed: {}", e); m["Sil_bio"] = np.nan        try:            adata_c = run_leiden_clustering(adata.copy(), emb_key, resolution=1.0)            m["NMI"] = compute_nmi(adata_c, bio_col)            m["ARI"] = compute_ari(adata_c, bio_col)        except Exception as e:            logger.warning("NMI/ARI failed: {}", e)            m["NMI"] = m["ARI"] = np.nan    else:        m["cLISI"] = m["Sil_bio"] = m["NMI"] = m["ARI"] = np.nan    # Aggregates (skip NaN)    batch_vals = [m[k] for k in ["kBET","iLISI","ASW_batch","graph_conn"] if not np.isnan(m.get(k, np.nan))]    bio_vals = [m[k] for k in ["cLISI","Sil_bio","NMI","ARI"] if not np.isnan(m.get(k, np.nan))]    m["AvgBATCH"] = geometric_mean(batch_vals) if batch_vals else np.nan    m["AvgBio"] = geometric_mean(bio_vals) if bio_vals else np.nan    return mdef simple_sil(emb, labels):    """Quick silhouette score, returns NaN if <2 groups."""    u = np.unique(labels)    if len(u) < 2: return np.nan    return silhouette_score(emb, labels)logger.info("Helpers defined")

## 6. Run Batch CorrectionProcess multi-batch cohorts (KIRC=5 batches, NSCLC=8 batches) with:1. **KL-cVAE** (baseline from notebook 4)2. **Adv2 adversarial** (two-optimizer, no MMD)3. **Adv2 mmd** (MMD only, no adversarial)4. **Adv2 both** (adversarial + MMD combined)

In [ ]:
# Cell 6: Run correctionsall_results = {}  # {cohort: {method: corrected_emb}}all_histories = {}  # {cohort: {method: history}}all_correctors = {}for cohort_name in MULTI_BATCH:    logger.info("\n" + "=" * 60 + "\nProcessing: {}\n" + "=" * 60, cohort_name)    adata = all_adata[cohort_name]    X_raw = np.asarray(adata.obsm[EMB_KEY]).astype(np.float32)    batches = adata.obs[BATCH_COL].astype(str).values    results = {"Raw": X_raw}    histories = {}    correctors = {}    # KL-cVAE baseline    logger.info("--- KL-cVAE ---")    kl_corr = CVAECorrector(KL_CONFIG)    results["KL-cVAE"] = kl_corr.fit_transform(X_raw, batches)    histories["KL-cVAE"] = kl_corr.history_    correctors["KL-cVAE"] = kl_corr    # Three Adv2 variants    for variant_name, cfg in ADV2_CONFIGS.items():        logger.info("--- {} ---", variant_name)        corr = CVAEAdv2Corrector(cfg)        results[variant_name] = corr.fit_transform(X_raw, batches)        histories[variant_name] = corr.history_        correctors[variant_name] = corr    all_results[cohort_name] = results    all_histories[cohort_name] = histories    all_correctors[cohort_name] = correctorslogger.info("All corrections complete!")

## 7. Training Curves

In [ ]:
# Cell 7: Plot training curvesfor cohort_name in MULTI_BATCH:    histories = all_histories[cohort_name]    fig, axes = plt.subplots(2, 3, figsize=(18, 10))    fig.suptitle(f"Training Curves: {cohort_name}", fontsize=14, fontweight="bold")    for i, (method, hist) in enumerate(histories.items()):        color = f"C{i}"        # Recon loss        axes[0, 0].plot(hist.recon, label=method, color=color, alpha=0.8)        axes[0, 0].set_title("Reconstruction (MSE)")        axes[0, 0].set_xlabel("Epoch")        if hasattr(hist, "disc_accuracy"):            # Disc accuracy            axes[0, 1].plot(hist.disc_accuracy, label=method, color=color, alpha=0.8)            axes[0, 1].set_title("Disc Accuracy")            axes[0, 1].axhline(1.0 / all_adata[cohort_name].obs[BATCH_COL].nunique(),                               ls="--", color="gray", alpha=0.5, label="random" if i==0 else None)        if hasattr(hist, "adv_enc"):            axes[0, 2].plot(hist.adv_enc, label=method, color=color, alpha=0.8)            axes[0, 2].set_title("Encoder Adv Loss")        if hasattr(hist, "adv_disc"):            axes[1, 0].plot(hist.adv_disc, label=method, color=color, alpha=0.8)            axes[1, 0].set_title("Disc CE Loss")        if hasattr(hist, "mmd"):            axes[1, 1].plot(hist.mmd, label=method, color=color, alpha=0.8)            axes[1, 1].set_title("MMD Loss")        if hasattr(hist, "lambda_schedule"):            axes[1, 2].plot(hist.lambda_schedule, label=method, color=color, alpha=0.8)            axes[1, 2].set_title("Lambda Schedule")    for ax in axes.flat:        ax.legend(fontsize=8)        ax.grid(True, alpha=0.3)    plt.tight_layout()    plt.show()

## 8. Before / After UMAP

In [ ]:
# Cell 8: UMAP visualizationfor cohort_name in MULTI_BATCH:    results = all_results[cohort_name]    adata = all_adata[cohort_name]    methods = list(results.keys())    n_methods = len(methods)    fig, axes = plt.subplots(2, n_methods, figsize=(5 * n_methods, 10))    fig.suptitle(f"UMAP: {cohort_name}", fontsize=14, fontweight="bold")    for j, method in enumerate(methods):        emb = results[method]        # Quick UMAP        tmp = ad.AnnData(obs=adata.obs.copy())        tmp.obsm["X_emb"] = emb        sc.pp.neighbors(tmp, use_rep="X_emb", n_neighbors=30)        sc.tl.umap(tmp)        umap_coords = tmp.obsm["X_umap"]        # Row 0: batch        batches = adata.obs[BATCH_COL].values        for b in np.unique(batches):            mask = batches == b            axes[0, j].scatter(umap_coords[mask, 0], umap_coords[mask, 1],                              s=5, alpha=0.5, label=b[:15])        axes[0, j].set_title(f"{method}\n(Batch)")        axes[0, j].legend(fontsize=5, loc="best", markerscale=2)        # Row 1: diagnosis        diags = adata.obs[DIAGNOSIS_COL].values        for d in np.unique(diags):            mask = diags == d            axes[1, j].scatter(umap_coords[mask, 0], umap_coords[mask, 1],                              s=5, alpha=0.5, label=d)        axes[1, j].set_title(f"{method}\n(Diagnosis)")        axes[1, j].legend(fontsize=7, loc="best", markerscale=2)    for ax in axes.flat:        ax.set_xticks([]); ax.set_yticks([])    plt.tight_layout()    plt.show()

## 9. Quick Silhouette Summary

In [ ]:
# Cell 9: Silhouette scoresrows = []for cohort_name in MULTI_BATCH:    results = all_results[cohort_name]    adata = all_adata[cohort_name]    batches = adata.obs[BATCH_COL].astype(str).values    diags = adata.obs[DIAGNOSIS_COL].astype(str).values    for method, emb in results.items():        rows.append({            "Cohort": cohort_name.split("_")[1] if "_" in cohort_name else cohort_name,            "Method": method,            "Sil_Batch": simple_sil(emb, batches),            "Sil_Diag": simple_sil(emb, diags),        })df_sil = pd.DataFrame(rows)df_sil["Batch_Δ"] = df_sil.groupby("Cohort")["Sil_Batch"].transform(lambda x: x - x.iloc[0])df_sil["Diag_Δ"] = df_sil.groupby("Cohort")["Sil_Diag"].transform(lambda x: x - x.iloc[0])display(df_sil.round(4))

## 10. Full scIB Metrics (per cohort)Metrics suite from [Luecken et al. 2022](https://www.biorxiv.org/content/10.1101/2023.04.30.538439v2):- **Batch mixing**: kBET, iLISI, ASW_batch, Graph Connectivity → **AvgBATCH**- **Bio conservation**: cLISI, Silhouette_bio, NMI, ARI → **AvgBio**

In [ ]:
# Cell 10: Full scIB metricsall_metrics = {}for cohort_name in MULTI_BATCH:    results = all_results[cohort_name]    adata = all_adata[cohort_name]    for method, emb in results.items():        key = f"{cohort_name[:4]}_{method}"        # Create temp AnnData with corrected embeddings        tmp = ad.AnnData(obs=adata.obs.copy())        tmp.obsm["X_emb"] = emb        logger.info("Computing metrics: {} / {}", cohort_name, method)        m = compute_all_metrics(tmp, "X_emb", BATCH_COL, DIAGNOSIS_COL)        m["Cohort"] = cohort_name        m["Method"] = method        all_metrics[key] = mdf_metrics = pd.DataFrame(all_metrics).Tcols_order = ["Cohort", "Method", "kBET", "iLISI", "ASW_batch", "graph_conn",              "AvgBATCH", "cLISI", "Sil_bio", "NMI", "ARI", "AvgBio"]cols_present = [c for c in cols_order if c in df_metrics.columns]df_metrics = df_metrics[cols_present]display(df_metrics.round(4))

## 11. Final Comparison Table

In [ ]:
# Cell 11: Comparison table per cohortfor cohort_name in MULTI_BATCH:    short = cohort_name.split("_")[1] if "_" in cohort_name else cohort_name    mask = df_metrics["Cohort"] == cohort_name    df_c = df_metrics[mask].set_index("Method").drop(columns=["Cohort"])    print(f"\n{'='*60}")    print(f"  {short}: Method Comparison")    print(f"{'='*60}")    display(df_c.round(4))    # Highlight best    for col in df_c.columns:        if col in ["AvgBATCH", "kBET", "iLISI", "ASW_batch", "graph_conn"]:            best = df_c[col].idxmax()            print(f"  Best {col}: {best} ({df_c.loc[best, col]:.4f})")        elif col in ["AvgBio", "cLISI", "Sil_bio", "NMI", "ARI"]:            best = df_c[col].idxmax()            print(f"  Best {col}: {best} ({df_c.loc[best, col]:.4f})")

## 12. Cross-Cohort Analysis (fixes NaN problem)Merge all 5 cohorts, use cohort name as batch label → all metrics computable.

In [ ]:
# Cell 12: Cross-cohort analysis# For Raw embeddings: merge all cohortsall_raw = []for name, adata in all_adata.items():    tmp = ad.AnnData(obs=adata.obs.copy())    tmp.obsm["X_raw"] = np.asarray(adata.obsm[EMB_KEY]).astype(np.float32)    tmp.obs["cohort_batch"] = name    all_raw.append(tmp)merged = ad.concat(all_raw, join="outer")logger.info("Merged: {} samples, {} cohort-batches", merged.n_obs, merged.obs["cohort_batch"].nunique())# Compute Raw metrics on mergedm_raw = compute_all_metrics(merged, "X_raw", "cohort_batch", DIAGNOSIS_COL)m_raw["Method"] = "Raw"# For corrected: we need to correct the merged data# Use best Adv2 variant on the combined dataX_merged = np.asarray(merged.obsm["X_raw"])batches_merged = merged.obs["cohort_batch"].astype(str).valuescross_results = {"Raw": m_raw}# KL-cVAE on mergedkl_merged = CVAECorrector(KL_CONFIG)emb_kl = kl_merged.fit_transform(X_merged, batches_merged)merged.obsm["X_kl"] = emb_klm_kl = compute_all_metrics(merged, "X_kl", "cohort_batch", DIAGNOSIS_COL)m_kl["Method"] = "KL-cVAE"cross_results["KL-cVAE"] = m_kl# Adv2 both on mergedadv2_merged = CVAEAdv2Corrector(CVAEAdv2Config(loss_type="both"))emb_adv2 = adv2_merged.fit_transform(X_merged, batches_merged)merged.obsm["X_adv2"] = emb_adv2m_adv2 = compute_all_metrics(merged, "X_adv2", "cohort_batch", DIAGNOSIS_COL)m_adv2["Method"] = "Adv2_both"cross_results["Adv2_both"] = m_adv2df_cross = pd.DataFrame(cross_results).Tprint("\n" + "=" * 60)print("  Cross-Cohort Comparison (5 cohorts merged)")print("=" * 60)display(df_cross.round(4))

## 13. Discriminator Convergence Diagnostic

In [ ]:
# Cell 13: Disc accuracy analysisfor cohort_name in MULTI_BATCH:    histories = all_histories[cohort_name]    n_batches = all_adata[cohort_name].obs[BATCH_COL].nunique()    random_baseline = 1.0 / n_batches    fig, ax = plt.subplots(figsize=(10, 4))    for method, hist in histories.items():        if hasattr(hist, "disc_accuracy") and hist.disc_accuracy:            ax.plot(hist.disc_accuracy, label=method, alpha=0.8)    ax.axhline(random_baseline, ls="--", color="red", alpha=0.7,               label=f"Random ({random_baseline:.2f})")    ax.set_title(f"Disc Accuracy: {cohort_name} (n_batches={n_batches})")    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")    ax.legend(); ax.grid(True, alpha=0.3)    ax.set_ylim(0, 1)    plt.tight_layout(); plt.show()    # Convergence check    for method, hist in histories.items():        if hasattr(hist, "disc_accuracy") and hist.disc_accuracy:            final_acc = np.mean(hist.disc_accuracy[-10:])            status = "✅ CONVERGED" if abs(final_acc - random_baseline) < 0.15 else "⚠️ NOT converged"            print(f"  {method}: final disc_acc={final_acc:.3f} {status}")

## 14. Summary & Next Steps### Expected outcomes:- **disc_accuracy → ~random** (1/n_batches) = adversarial working- **AvgBATCH ↑** = better batch mixing- **AvgBio ↑** (or stable) = biology preserved- **Adv2_both** should balance batch mixing and bio preservation better than KL-cVAE